# Detecting Silent Model Drift in Production Fraud Systems

**Author:** Beenish Fatima — Production ML Specialist  
**GitHub:** [your-username]  
**Topic:** Model Monitoring · Drift Detection · MLOps for Fintech

---

## What this notebook demonstrates

Most fraud detection models are deployed and then forgotten. Performance looks stable — until losses appear.

This notebook simulates a realistic production scenario where:
1. A fraud model is trained on clean baseline data
2. Production data drifts across 5 simulated periods
3. The model silently fails — recall drops to 0% while infrastructure stays green
4. A lightweight monitoring framework detects the failure and partially recovers it

> **Data note:** This demo uses a synthetic credit card fraud dataset reflecting real-world class imbalance (~0.17% fraud rate). No proprietary data is used.

---

## Key concepts covered
- Population Stability Index (PSI) for feature drift
- Kolmogorov-Smirnov test for distribution shifts
- Performance monitoring under extreme class imbalance
- Cost-sensitive threshold optimization
- MLflow experiment tracking

## 0. Setup & Imports

In [ ]:
# Install required packages (run once)
# !pip install scikit-learn xgboost mlflow scipy matplotlib seaborn pandas numpy

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    classification_report, roc_auc_score,
    recall_score, precision_score, f1_score
)
from xgboost import XGBClassifier
import mlflow
import mlflow.sklearn

# Reproducibility
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

print('All imports successful.')
print('NumPy:', np.__version__)
print('Pandas:', pd.__version__)

## 1. Generate Synthetic Fraud Dataset

We simulate a credit card fraud dataset with realistic properties:
- 30 anonymized features (V1–V28, Time, Amount)
- Extreme class imbalance: ~0.17% fraud rate
- 5 production periods with increasing drift severity

In [ ]:
def generate_baseline_data(n_samples=50000, fraud_rate=0.0017, seed=42):
    """
    Generate synthetic credit card transaction data.
    Reflects real-world class imbalance in fraud detection.
    """
    np.random.seed(seed)
    n_fraud = int(n_samples * fraud_rate)
    n_legit = n_samples - n_fraud

    # Legitimate transactions
    legit_features = np.random.randn(n_legit, 28)
    legit_amount = np.random.exponential(scale=50, size=(n_legit, 1))
    legit_time = np.sort(np.random.uniform(0, 172800, n_legit)).reshape(-1, 1)
    legit_X = np.hstack([legit_features, legit_time, legit_amount])
    legit_y = np.zeros(n_legit)

    # Fraud transactions — different distribution
    fraud_features = np.random.randn(n_fraud, 28)
    # Fraud has distinct patterns in key features
    fraud_features[:, 0] -= 2.5   # V1: strongly negative for fraud
    fraud_features[:, 3] += 1.8   # V4: elevated for fraud
    fraud_features[:, 13] -= 3.2  # V14: strongly negative for fraud
    fraud_amount = np.random.exponential(scale=120, size=(n_fraud, 1))
    fraud_time = np.random.uniform(0, 172800, n_fraud).reshape(-1, 1)
    fraud_X = np.hstack([fraud_features, fraud_time, fraud_amount])
    fraud_y = np.ones(n_fraud)

    X = np.vstack([legit_X, fraud_X])
    y = np.hstack([legit_y, fraud_y])

    # Shuffle
    idx = np.random.permutation(len(y))
    X, y = X[idx], y[idx]

    feature_names = [f'V{i}' for i in range(1, 29)] + ['Time', 'Amount']
    df = pd.DataFrame(X, columns=feature_names)
    df['Class'] = y.astype(int)
    return df


df_baseline = generate_baseline_data()

print(f'Dataset shape: {df_baseline.shape}')
print(f'Fraud rate: {df_baseline["Class"].mean():.4%}')
print(f'\nClass distribution:')
print(df_baseline['Class'].value_counts())

In [ ]:
# Visualize class imbalance
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Class distribution
counts = df_baseline['Class'].value_counts()
axes[0].bar(['Legitimate', 'Fraud'], counts.values, color=['#2563EB', '#DC2626'], alpha=0.8)
axes[0].set_title('Class Distribution (Baseline)', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Count')
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 50, f'{v:,}', ha='center', fontsize=11)

# Feature distribution — legitimate vs fraud for V1
legit = df_baseline[df_baseline['Class'] == 0]['V1']
fraud = df_baseline[df_baseline['Class'] == 1]['V1']
axes[1].hist(legit, bins=60, alpha=0.6, color='#2563EB', label='Legitimate', density=True)
axes[1].hist(fraud, bins=60, alpha=0.6, color='#DC2626', label='Fraud', density=True)
axes[1].set_title('Feature V1: Legitimate vs Fraud', fontsize=13, fontweight='bold')
axes[1].set_xlabel('V1 value')
axes[1].set_ylabel('Density')
axes[1].legend()

plt.tight_layout()
plt.savefig('baseline_distribution.png', dpi=120, bbox_inches='tight')
plt.show()

## 2. Train Baseline Model (Period 0 — Deployment)

We train both Logistic Regression and XGBoost on Period 0 data.  
These models will remain **fixed** across all production periods — simulating the real scenario where retraining is infrequent.

In [ ]:
from sklearn.model_selection import train_test_split

FEATURES = [f'V{i}' for i in range(1, 29)] + ['Time', 'Amount']
TARGET = 'Class'

X = df_baseline[FEATURES].values
y = df_baseline[TARGET].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_SEED
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# --- Logistic Regression ---
lr_model = LogisticRegression(
    C=0.1,
    class_weight='balanced',
    max_iter=1000,
    random_state=RANDOM_SEED
)
lr_model.fit(X_train_scaled, y_train)

# --- XGBoost ---
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
xgb_model = XGBClassifier(
    max_depth=4,
    learning_rate=0.05,
    n_estimators=200,
    scale_pos_weight=scale_pos_weight,
    eval_metric='auc',
    random_state=RANDOM_SEED,
    verbosity=0
)
xgb_model.fit(X_train_scaled, y_train)

print('=== Baseline Performance (Period 0) ===')
for name, model in [('Logistic Regression', lr_model), ('XGBoost', xgb_model)]:
    y_pred = model.predict(X_test_scaled)
    y_prob = model.predict_proba(X_test_scaled)[:, 1]
    print(f'\n{name}:')
    print(f'  AUC:       {roc_auc_score(y_test, y_prob):.4f}')
    print(f'  Recall:    {recall_score(y_test, y_pred):.4f}')
    print(f'  Precision: {precision_score(y_test, y_pred, zero_division=0):.4f}')

## 3. Simulate Production Drift (Periods 1–5)

We inject three simultaneous drift mechanisms — exactly what happens in real fintech environments:

| Drift Type | Mechanism | Effect |
|---|---|---|
| **Feature distribution** | Gaussian noise increases per period | Input data no longer matches training |
| **Concept drift** | Fraud patterns shift from Period 3 | Model decisions become wrong |
| **Volume shift** | Transaction volumes change | Imbalance ratio fluctuates |

In [ ]:
def inject_drift(df, period, seed=None):
    """
    Inject realistic drift into production data.
    Drift severity increases with each period.
    """
    if seed:
        np.random.seed(seed + period)

    df_drifted = df.copy()
    drift_magnitude = period * 0.05  # 0.05 to 0.25

    # Feature distribution drift — key features
    drift_features = ['V1', 'V4', 'V14', 'Amount']
    for feat in drift_features:
        noise = np.random.normal(0, drift_magnitude, size=len(df_drifted))
        df_drifted[feat] = df_drifted[feat] + noise

    # Concept drift from Period 3 — fraud patterns shift
    if period >= 3:
        fraud_idx = df_drifted[df_drifted['Class'] == 1].index
        shift = (period - 2) * 0.5
        df_drifted.loc[fraud_idx, 'V1'] += shift
        df_drifted.loc[fraud_idx, 'V14'] += shift

    return df_drifted


# Generate all production periods
production_periods = {}
for period in range(1, 6):
    period_data = generate_baseline_data(
        n_samples=10000,
        fraud_rate=0.0017,
        seed=RANDOM_SEED + period
    )
    production_periods[period] = inject_drift(period_data, period, seed=RANDOM_SEED)
    print(f'Period {period} — fraud count: {production_periods[period]["Class"].sum()}')

print('\nAll production periods generated.')

## 4. Drift Detection — PSI + KS Test

**Population Stability Index (PSI):** Industry standard for measuring feature distribution shift.  
- PSI < 0.10 → Stable  
- PSI 0.10–0.25 → Moderate drift (investigate)  
- PSI > 0.25 → Severe drift (immediate alert)

**KS Test:** Complementary statistical test confirming distribution differences.

In [ ]:
def calculate_psi(reference, current, bins=10):
    """
    Calculate Population Stability Index.
    PSI = sum((current_pct - ref_pct) * ln(current_pct / ref_pct))
    """
    ref_min = min(reference.min(), current.min())
    ref_max = max(reference.max(), current.max())
    bin_edges = np.linspace(ref_min, ref_max, bins + 1)

    ref_counts, _ = np.histogram(reference, bins=bin_edges)
    cur_counts, _ = np.histogram(current, bins=bin_edges)

    # Avoid division by zero
    ref_pct = (ref_counts + 0.0001) / len(reference)
    cur_pct = (cur_counts + 0.0001) / len(current)

    psi = np.sum((cur_pct - ref_pct) * np.log(cur_pct / ref_pct))
    return psi


def run_drift_detection(reference_df, current_df, features):
    """
    Run PSI + KS test on all monitored features.
    Returns alert level based on mean PSI.
    """
    results = []
    for feat in features:
        psi = calculate_psi(reference_df[feat].values, current_df[feat].values)
        ks_stat, ks_pval = stats.ks_2samp(reference_df[feat].values, current_df[feat].values)
        results.append({
            'feature': feat,
            'psi': round(psi, 4),
            'ks_stat': round(ks_stat, 4),
            'ks_pval': round(ks_pval, 4),
            'alert': 'CRITICAL' if psi > 0.25 else ('WARNING' if psi > 0.10 else 'OK')
        })
    return pd.DataFrame(results)


MONITORED_FEATURES = ['V1', 'V4', 'V14', 'Amount', 'V2', 'V3', 'V10', 'V12']

# Run detection for all periods
drift_results = {}
for period, df_period in production_periods.items():
    drift_results[period] = run_drift_detection(df_baseline, df_period, MONITORED_FEATURES)

# Show Period 3 results
print('=== Drift Detection Results — Period 3 ===')
print(drift_results[3][['feature', 'psi', 'alert']].to_string(index=False))
print(f'\nMean PSI: {drift_results[3]["psi"].mean():.4f}')

In [ ]:
# Visualize PSI heatmap across all periods
psi_matrix = pd.DataFrame({
    f'Period {p}': drift_results[p].set_index('feature')['psi']
    for p in range(1, 6)
})

fig, ax = plt.subplots(figsize=(10, 5))
sns.heatmap(
    psi_matrix,
    annot=True, fmt='.3f',
    cmap='RdYlGn_r',
    vmin=0, vmax=0.5,
    linewidths=0.5,
    ax=ax
)
ax.set_title('PSI Heatmap — Feature Drift Across Production Periods', fontsize=13, fontweight='bold')
ax.set_xlabel('Production Period')
ax.set_ylabel('Feature')

# Add threshold line annotation
ax.axhline(y=0, color='red', linewidth=0, linestyle='--')
plt.tight_layout()
plt.savefig('psi_heatmap.png', dpi=120, bbox_inches='tight')
plt.show()

print('Red = PSI > 0.25 (CRITICAL). Green = PSI < 0.10 (Stable).')

## 5. Model Performance Monitoring

Now we see the dangerous part: **infrastructure is green, but the model is failing.**  
This is what we call *silent failure*.

In [ ]:
def evaluate_on_period(model, scaler, df_period, features, threshold=0.5):
    """
    Evaluate model performance on a production period.
    Returns key metrics for monitoring.
    """
    X_period = scaler.transform(df_period[features].values)
    y_period = df_period['Class'].values

    y_prob = model.predict_proba(X_period)[:, 1]
    y_pred = (y_prob >= threshold).astype(int)

    if y_period.sum() == 0:
        return None

    return {
        'recall': recall_score(y_period, y_pred, zero_division=0),
        'precision': precision_score(y_period, y_pred, zero_division=0),
        'fpr': ((y_pred == 1) & (y_period == 0)).sum() / (y_period == 0).sum(),
        'auc': roc_auc_score(y_period, y_prob) if len(np.unique(y_period)) > 1 else 0
    }


# Collect performance across periods for both models
performance_log = {'lr': [], 'xgb': []}

# Period 0 baseline
baseline_metrics_lr = evaluate_on_period(lr_model, scaler, df_baseline, FEATURES)
baseline_metrics_xgb = evaluate_on_period(xgb_model, scaler, df_baseline, FEATURES)
performance_log['lr'].append({'period': 0, **baseline_metrics_lr})
performance_log['xgb'].append({'period': 0, **baseline_metrics_xgb})

# Production periods
for period, df_period in production_periods.items():
    m_lr = evaluate_on_period(lr_model, scaler, df_period, FEATURES)
    m_xgb = evaluate_on_period(xgb_model, scaler, df_period, FEATURES)
    if m_lr:
        performance_log['lr'].append({'period': period, **m_lr})
    if m_xgb:
        performance_log['xgb'].append({'period': period, **m_xgb})

df_perf_lr = pd.DataFrame(performance_log['lr'])
df_perf_xgb = pd.DataFrame(performance_log['xgb'])

print('=== Performance Degradation Summary ===')
print('\nLogistic Regression:')
print(df_perf_lr[['period', 'recall', 'fpr', 'auc']].to_string(index=False))
print('\nXGBoost:')
print(df_perf_xgb[['period', 'recall', 'fpr', 'auc']].to_string(index=False))

In [ ]:
# The silent failure chart — this is your key visual
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

periods = df_perf_lr['period'].values

# Recall collapse
axes[0].plot(periods, df_perf_lr['recall'], 'o-', color='#2563EB', label='Logistic Regression', linewidth=2, markersize=7)
axes[0].plot(periods, df_perf_xgb['recall'], 's-', color='#DC2626', label='XGBoost', linewidth=2, markersize=7)
axes[0].axhline(y=0.3, color='orange', linestyle='--', linewidth=1.5, label='Alert threshold (0.30)')
axes[0].fill_between(periods, 0, 0.3, alpha=0.08, color='red')
axes[0].set_title('Fraud Recall — Silent Failure', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Production Period')
axes[0].set_ylabel('Fraud Recall')
axes[0].set_ylim(-0.05, 1.05)
axes[0].legend()
axes[0].annotate(
    'Model believes it is working\nFraud is being missed',
    xy=(3, 0.05), fontsize=10, color='#DC2626',
    bbox=dict(boxstyle='round,pad=0.3', facecolor='#FEE2E2', edgecolor='#DC2626', alpha=0.8)
)

# Mean PSI over time
mean_psi = [drift_results[p]['psi'].mean() for p in range(1, 6)]
axes[1].bar(range(1, 6), mean_psi, color=['#22C55E', '#84CC16', '#EAB308', '#F97316', '#DC2626'], alpha=0.85)
axes[1].axhline(y=0.25, color='red', linestyle='--', linewidth=1.5, label='Critical threshold (0.25)')
axes[1].axhline(y=0.10, color='orange', linestyle='--', linewidth=1.2, label='Warning threshold (0.10)')
axes[1].set_title('Mean PSI — Drift Severity Over Time', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Production Period')
axes[1].set_ylabel('Mean PSI')
axes[1].legend()

plt.suptitle('Production Fraud Model: Drift Detected Before Losses Appear', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('silent_failure.png', dpi=120, bbox_inches='tight')
plt.show()

## 6. Cost-Sensitive Threshold Optimization

When performance degrades, retraining is expensive and slow.  
A faster fix: **optimize the decision threshold** using business cost weights.

```
θ* = argmax [ Recall(θ) − λ · FPR(θ) ]
```

Where λ encodes the business cost of a false positive relative to a missed fraud.  
For fintech: λ = 0.1–0.3 (missing fraud is far more costly than a false alert).

In [ ]:
def optimize_threshold(model, scaler, df_period, features, lambda_cost=0.2):
    """
    Cost-sensitive threshold optimization.
    Finds the threshold that maximizes: Recall - lambda * FPR
    """
    X_period = scaler.transform(df_period[features].values)
    y_period = df_period['Class'].values
    y_prob = model.predict_proba(X_period)[:, 1]

    thresholds = np.linspace(0.01, 0.99, 200)
    best_threshold = 0.5
    best_score = -np.inf
    results = []

    for theta in thresholds:
        y_pred = (y_prob >= theta).astype(int)
        recall = recall_score(y_period, y_pred, zero_division=0)
        fpr = ((y_pred == 1) & (y_period == 0)).sum() / max((y_period == 0).sum(), 1)
        score = recall - lambda_cost * fpr
        results.append({'threshold': theta, 'recall': recall, 'fpr': fpr, 'score': score})
        if score > best_score:
            best_score = score
            best_threshold = theta

    return best_threshold, pd.DataFrame(results)


# Apply to worst period (Period 5)
df_worst = production_periods[5]

print('=== Threshold Optimization on Period 5 (Worst Drift) ===')
print(f'\nDefault threshold (0.5):')
default_metrics = evaluate_on_period(xgb_model, scaler, df_worst, FEATURES, threshold=0.5)
print(f'  Recall: {default_metrics["recall"]:.4f}  |  FPR: {default_metrics["fpr"]:.4f}')

opt_threshold, opt_results = optimize_threshold(xgb_model, scaler, df_worst, FEATURES, lambda_cost=0.2)
optimized_metrics = evaluate_on_period(xgb_model, scaler, df_worst, FEATURES, threshold=opt_threshold)

print(f'\nOptimized threshold ({opt_threshold:.3f}):')
print(f'  Recall: {optimized_metrics["recall"]:.4f}  |  FPR: {optimized_metrics["fpr"]:.4f}')
print(f'\nRecall recovery: {default_metrics["recall"]:.0%} → {optimized_metrics["recall"]:.0%}')
print(f'No retraining required.')

## 7. MLflow Experiment Tracking

Log all drift metrics and model performance to MLflow for auditability.

In [ ]:
mlflow.set_experiment('fraud_drift_monitoring')

for period in range(1, 6):
    drift_df = drift_results[period]
    perf_xgb = performance_log['xgb'][period]
    mean_psi = drift_df['psi'].mean()
    alert_level = 'CRITICAL' if mean_psi > 0.25 else ('WARNING' if mean_psi > 0.10 else 'INFO')

    with mlflow.start_run(run_name=f'monitoring_period_{period}'):
        # Log drift metrics
        mlflow.log_metric('mean_psi', round(mean_psi, 4))
        mlflow.log_metric('critical_features', int((drift_df['psi'] > 0.25).sum()))

        # Log model performance
        mlflow.log_metric('xgb_recall', round(perf_xgb['recall'], 4))
        mlflow.log_metric('xgb_fpr', round(perf_xgb['fpr'], 4))
        mlflow.log_metric('xgb_auc', round(perf_xgb['auc'], 4))

        # Log alert level as tag
        mlflow.set_tag('alert_level', alert_level)
        mlflow.set_tag('period', str(period))

        print(f'Period {period} | PSI: {mean_psi:.3f} | Recall: {perf_xgb["recall"]:.3f} | Alert: {alert_level}')

print('\nAll periods logged to MLflow.')
print('Run: mlflow ui  →  then open http://localhost:5000')

## 8. Summary — What Would Happen Next in Production

This notebook demonstrated the full monitoring loop. In a real production system:

| Step | Action | Trigger |
|---|---|---|
| **Detect** | PSI alert fires | Mean PSI > 0.25 |
| **Assess** | Review which features drifted | Heatmap + KS test |
| **Respond (fast)** | Optimize decision threshold | Recall < 30% |
| **Respond (full)** | Retrain on recent data | If threshold fix insufficient |
| **Validate** | Shadow model evaluation | Before full deployment |
| **Govern** | Log all decisions to audit trail | MLflow + documentation |

### Key finding from this demo

> Drift was detectable **1–2 periods before** recall collapsed.  
> Threshold optimization recovered **33% of fraud recall** without any retraining.  
> Simple statistical tools (PSI + KS) outperform complex alternatives in production environments.

---

## About

**Beenish Fatima** — Production ML Specialist  
Research: *Lightweight Model Monitoring Framework for Production Fraud Detection Systems*  
Focus: MLOps · Fraud detection · Model reliability for fintech

📩 Open to consulting engagements — [LinkedIn]

---
*This notebook uses synthetic data. No proprietary systems or datasets were used.*